In [ ]:
%pip install torch torchvision torchaudio torchcodec wandb pandas tqdm scikit-learn

In [ ]:
# CNN
import torch.nn as nn
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

class EfficientBirbNN(nn.Module):
    # It is 234 according to the competition description
    def __init__(self, num_classes = 234, pretrained=True):
        super().__init__()
        
        # 1. Load the base EfficientNet model
        weights = EfficientNet_B3_Weights.DEFAULT if pretrained else None
        self.base_model = efficientnet_b3(weights=weights)
        
        # 2. Modify the first convolutional layer to accept 1-channel spectrograms
        # EfficientNet's first layer is located at self.base_model.features[0][0]
        original_conv = self.base_model.features[0][0]
        self.base_model.features[0][0] = nn.Conv2d(
            in_channels=1, 
            out_channels=original_conv.out_channels, 
            kernel_size=original_conv.kernel_size, 
            stride=original_conv.stride, 
            padding=original_conv.padding, 
            bias=False
        )
                
        # 3. Modify the final classification layer for your specific number of bird classes
        in_features = self.base_model.classifier[1].in_features
        self.base_model.classifier[1] = nn.Sequential(
            nn.Dropout(p=0.4), # Extra dropout to prevent overfitting on audio data
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)


In [ ]:
import torchaudio
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

class SoundscapeDataset(Dataset):
    def __init__(self, audio_path, clip_length=5.0, sample_rate=32000):
        self.clip_length = clip_length
        self.sample_rate = sample_rate
        self.chunk_size = int(self.sample_rate * self.clip_length)
        
        # Load the ENTIRE audio file into RAM. 
        # A 10-min file at 32kHz is only ~75MB, so this is perfectly safe and much faster than seeking.
        self.waveform, sr = torchaudio.load(audio_path)
        
        # Failsafe: Resample if the soundscape isn't 32kHz
        if sr != self.sample_rate:
            self.waveform = torchaudio.functional.resample(self.waveform, sr, self.sample_rate)
            
        # Failsafe: Convert to mono if stereo
        if self.waveform.shape[0] > 1:
            self.waveform = self.waveform.mean(dim=0, keepdim=True)
            
        # Drop the trailing audio that doesn't fit perfectly into a 5-sec window
        self.num_chunks = self.waveform.shape[1] // self.chunk_size
        
        # WE MUST USE THE EXACT SAME TRANSFORMS AS TRAINING
        self.amp_to_db = torchaudio.transforms.AmplitudeToDB(stype='power')
        self.mel_spect = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sample_rate, n_fft=800, n_mels=64
        )

    def __len__(self):
        return self.num_chunks

    def __getitem__(self, idx):
        start_idx = idx * self.chunk_size
        end_idx   = start_idx + self.chunk_size
        
        chunk = self.waveform[:, start_idx:end_idx]
        
        # Generate and Standardize Spectrogram
        spectrogram = self.mel_spect(chunk)
        spectrogram = self.amp_to_db(spectrogram)
        mean, std   = spectrogram.mean(), spectrogram.std() + 1e-6
        spectrogram = (spectrogram - mean) / std
        
        # Calculate the end time of this chunk (5, 10, 15... etc.)
        end_time = (idx + 1) * int(self.clip_length)
        
        return spectrogram, end_time

In [ ]:
import os
import glob
import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from torch.amp import autocast
from tqdm import tqdm

@torch.no_grad()
def generate_soundscape_predictions(model, soundscapes_dir, label_to_idx, device='cuda'):
    """
    Processes a full folder of soundscapes and generates a Kaggle-formatted submission DataFrame.
    """
    model.eval()
    
    # Get all audio files in the folder
    audio_files = glob.glob(os.path.join(soundscapes_dir, "*.ogg"))
    if not audio_files:
        raise FileNotFoundError(f"No .ogg files found in {soundscapes_dir}")
        
    print(f"Found {len(audio_files)} soundscapes. Running inference...")
    
    # Create the column names exactly as Kaggle expects (sorted alphabetically)
    # The reverse mapping helps us build the columns correctly
    idx_to_label = {v: k for k, v in label_to_idx.items()}
    sorted_labels = [idx_to_label[i] for i in range(len(idx_to_label))]
    
    all_rows = []
    
    for audio_path in tqdm(audio_files, desc="Processing Soundscapes"):
        # Extract the soundscape ID (e.g., "10534_COR" from "10534_COR.ogg")
        file_id = os.path.basename(audio_path).split('.')[0]
        
        # Re-use the SoundscapeDataset from the previous step
        dataset = SoundscapeDataset(audio_path, clip_length=5.0)
        loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
        
        for spectrograms, end_times in loader:
            spectrograms = spectrograms.to(device, non_blocking=True)
            
            with autocast('cuda'):
                logits = model(spectrograms)
                probs = torch.sigmoid(logits).cpu().numpy()
            
            end_times = end_times.numpy()
            
            # Build the rows for this batch
            for i in range(len(probs)):
                row_id = f"{file_id}_{end_times[i]}"
                
                # Create a dictionary for this row: {'row_id': '...', 'acafly': 0.01, 'acowoo': 0.05, ...}
                row_data = {'row_id': row_id}
                for class_idx, prob in enumerate(probs[i]):
                    row_data[idx_to_label[class_idx]] = prob
                    
                all_rows.append(row_data)
                
    # Compile into a DataFrame
    submission_df = pd.DataFrame(all_rows)
    
    # Ensure columns are ordered: row_id, then alphabetical bird labels
    cols = ['row_id'] + sorted_labels
    submission_df = submission_df[cols]
    
    return submission_df

In [ ]:
from sklearn.metrics import roc_auc_score

def time_to_seconds(time_str):
    """Converts a timestamp like '00:00:05' into integer seconds (5)."""
    h, m, s = map(int, str(time_str).split(':'))
    return h * 3600 + m * 60 + s

def evaluate_submission(submission_df, ground_truth_path):
    """
    Parses the raw Kaggle soundscape labels and compares them against predictions.
    """
    print("\nLoading and parsing ground truth labels...")
    raw_gt_df = pd.read_csv(ground_truth_path)
    
    # 1. Reconstruct the row_id to match the predictions (e.g., "filename_5")
    file_ids = raw_gt_df['filename'].str.replace('.ogg', '', regex=False)
    end_secs = raw_gt_df['end'].apply(time_to_seconds)
    raw_gt_df['row_id'] = file_ids + '_' + end_secs.astype(str)
    
    # 2. Identify the bird classes we are predicting
    bird_columns = [c for c in submission_df.columns if c != 'row_id']
    
    # 3. Build a one-hot encoded matrix for the ground truth
    # Start with all zeros
    gt_matrix = pd.DataFrame(0, index=range(len(raw_gt_df)), columns=bird_columns)
    gt_matrix['row_id'] = raw_gt_df['row_id'].values
    
    # Populate the 1s for the birds actually present
    for idx, row in raw_gt_df.iterrows():
        # Handle cases where there might be 'nocall' or NaN
        labels_str = str(row['primary_label'])
        if labels_str.lower() != 'nocall' and labels_str != 'nan':
            # Split the semicolon-separated IDs
            active_birds = labels_str.split(';')
            for bird in active_birds:
                bird = bird.strip()
                if bird in bird_columns:
                    gt_matrix.at[idx, bird] = 1.0
                
    # 4. Align the dataframes (critical if any rows are missing or out of order)
    print("Aligning predictions with ground truth...")
    merged = pd.merge(gt_matrix, submission_df, on='row_id', suffixes=('_true', '_pred'))
    
    if len(merged) == 0:
        return "Error: Could not match any row_ids between predictions and ground truth."
    
    # 5. Extract aligned true and predicted matrices
    y_true = merged[[f"{c}_true" for c in bird_columns]].values
    y_pred = merged[[f"{c}_pred" for c in bird_columns]].values
    
    # 6. Metric Safeguard: Only score classes that actually appear in this evaluation set
    # ROC-AUC requires at least one positive (1) and one negative (0) sample per class.
    valid_classes = np.any(y_true == 1, axis=0) & np.any(y_true == 0, axis=0)
    
    if not np.any(valid_classes):
        return "Error: No valid classes in ground truth to evaluate (no positive samples)."
        
    print(f"Scoring {valid_classes.sum()} valid classes...")
    
    # 7. Calculate Macro ROC-AUC
    macro_auc = roc_auc_score(
        y_true[:, valid_classes], 
        y_pred[:, valid_classes], 
        average='macro'
    )
    
    return macro_auc

In [ ]:
import wandb

# --- Path Configuration ---
root_path = os.path.join("..", "birdclef-2026")
SOUNDSCAPES_DIR = os.path.join(root_path, "train_soundscapes")
LABELS_CSV      = os.path.join(root_path, "train_soundscapes_labels.csv")

# 1. Build taxonomical dictionary mapping
unique_labels = pd.read_csv(os.path.join(root_path, "taxonomy.csv"))
unique_labels_series = sorted(unique_labels['primary_label'].unique())
master_label_to_idx = {label: i for i, label in enumerate(unique_labels_series)}
num_classes         = len(master_label_to_idx)

# Define the specific versions or aliases you want to compare
ARTIFACT_VERSIONS = ["v6", "v8", "v10", "v12", "v14"] 
PROJECT_PATH = "pumpkin_person-tu-dresden/CNN-Birds"
ARTIFACT_NAME = "cnn_bird_model_specaugment"

if __name__ == "__main__":
    # Device setup for Linux ThinkPad execution
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running evaluation pipeline on target device: {device}")
    
    # Initialize a central evaluation run in WandB to log the comparison results
    comparison_run = wandb.init(
        project="CNN-Birds",
        name="Model-Evaluation",
        job_type="evaluation",
        notes="Comparing soundscape Macro ROC-AUC scores across multiple WandB artifact checkpoints."
    )
    
    # Initialize a tracking table to aggregate results side-by-side
    columns = ["Artifact Version", "Soundscape Macro ROC-AUC"]
    results_table = wandb.Table(columns=columns)
    
    # Initialize WandB API client for artifact retrieval
    api = wandb.Api()
    
    # Loop over all requested versions
    for version in ARTIFACT_VERSIONS:
        print(f"\n{'-'*50}\nRetrieving and evaluating artifact version: {version}\n{'-'*50}")
        
        try:
            # Construct full artifact URI string
            artifact_uri = f"{PROJECT_PATH}/{ARTIFACT_NAME}:{version}"
            artifact = api.artifact(artifact_uri)
            artifact_dir = artifact.download()
            
            # Re-instantiate a fresh model architecture shell
            model = EfficientBirbNN(num_classes=num_classes)
            
            # Locate checkpoint file inside the dynamically created artifact download folder
            checkpoint_path = os.path.join(artifact_dir, "best_efficientbirb_model.pth")
            
            if not os.path.exists(checkpoint_path):
                # Fallback safeguard check in case the checkpoint was logged under a generic fallback name
                pth_files = glob.glob(os.path.join(artifact_dir, "*.pth"))
                if pth_files:
                    checkpoint_path = pth_files[0]
                else:
                    raise FileNotFoundError(f"No .pth file found in downloaded artifact directory for {version}")
            
            # Safely map checkpoint weights to local ThinkPad hardware layout
            checkpoint = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            model.to(device)
            
            # Run inference loop
            print(f"Starting soundscape inference for version {version}...")
            submission_df = generate_soundscape_predictions(
                model=model, 
                soundscapes_dir=SOUNDSCAPES_DIR, 
                label_to_idx=master_label_to_idx, 
                device=device
            )
            
            # Evaluate against the ground truth metrics if available
            if os.path.exists(LABELS_CSV):
                final_score = evaluate_submission(submission_df, LABELS_CSV)
                print(f"🔥 [{version}] SOUNDSCAPE MACRO ROC-AUC: {final_score:.4f}")
                
                # Append data row to the local WandB table representation
                results_table.add_data(version, final_score)
                
                # Log the individual metric point tied to this specific run
                wandb.log({f"macro_roc_auc_{version}": final_score})
            else:
                print(f"Ground truth labels missing at {LABELS_CSV}. Evaluation skipped for {version}.")
                results_table.add_data(version, None)
                
        except Exception as e:
            print(f"❌ Failed processing checkpoint version {version}. Error: {e}")
            continue

    # Commit the summary table matrix back to the active WandB dashboard run
    wandb.log({"Artifact Performance Matrix": results_table})
    comparison_run.finish()
    print("\nEvaluation run complete. Results successfully synced to your Weights & Biases dashboard.")